# Initial Multi-Model Paper Metrics Benchmark

1. Load validation articles from XSUM and CNN/DailyMail
2. Generate summaries with multiple models
3. Save generated summaries as JSONL
4. Compute ROUGE-1, ROUGE-2, ROUGE-L, METEOR, and BERTScore

Follows the paper's setup: XSUM uses one sentence, while CNN/DailyMail uses three sentences.

In [23]:
from pathlib import Path
from datetime import datetime
import json
import os
import time

import pandas as pd
import requests
from bert_score import score as bert_score
from datasets import load_dataset
from nltk.translate.meteor_score import single_meteor_score
from rouge_score import rouge_scorer

## Configuration

In [24]:
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "spec.md").exists() and (PROJECT_ROOT.parent / "spec.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

def load_env_file(env_file):
    if not env_file.exists():
        return

    for line in env_file.read_text(encoding="utf-8").splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue

        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


load_env_file(PROJECT_ROOT / ".env")

# Change this number to control how many articles are sampled from each dataset.
# Total generations = NUM_EXAMPLES_PER_DATASET * number of datasets * number of models.
NUM_EXAMPLES_PER_DATASET = 100

# Each run gets a timestamp so results do not overwrite previous runs.
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S")

OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_TAGS_URL = "http://localhost:11434/api/tags"
OPENAI_URL = "https://api.openai.com/v1/responses"
TEMPERATURE = 0.3

MODEL_CONFIGS = [
    {"provider": "ollama", "model": "llama3.1:latest"},
    {"provider": "ollama", "model": "gemma2:9b"},
    {"provider": "openai", "model": os.environ.get("OPENAI_MODEL", "gpt-4o-mini")},
]

CNN_DM_VALIDATION_FILE = PROJECT_ROOT / "data" / "cnn_dailymail_kaggle" / "cnn_dailymail" / "validation.csv"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
RESULTS_DIR = PROJECT_ROOT / "results"
OUTPUT_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

OUTPUT_FILE = OUTPUT_DIR / f"{RUN_ID}_initial_summaries.jsonl"
RESULTS_FILE = RESULTS_DIR / f"{RUN_ID}_initial_metric_scores.csv"
COMBINED_RESULTS_FILE = RESULTS_DIR / f"{RUN_ID}_initial_summaries_with_metrics.csv"

print(f"Run ID: {RUN_ID}")
print(f"Examples per dataset: {NUM_EXAMPLES_PER_DATASET}")
print("Models:")
for model_config in MODEL_CONFIGS:
    print(f"- {model_config['provider']} / {model_config['model']}")

PROJECT_ROOT

Run ID: 20260519_213553
Examples per dataset: 100
Models:
- ollama / llama3.1:latest
- ollama / gemma2:9b
- openai / gpt-4o-mini-2024-07-18


WindowsPath('C:/Users/fangw/162 proj')

## Verify Model Setup

If this cell fails, open Ollama first. If an Ollama model is missing, pull it in Git Bash. The OpenAI model requires `OPENAI_API_KEY` in `.env`.

In [25]:
ollama_response = requests.get(OLLAMA_TAGS_URL, timeout=10)
ollama_response.raise_for_status()

installed_ollama_models = [model["name"] for model in ollama_response.json().get("models", [])]
required_ollama_models = [
    model_config["model"]
    for model_config in MODEL_CONFIGS
    if model_config["provider"] == "ollama"
]
missing_ollama_models = [
    model_name
    for model_name in required_ollama_models
    if model_name not in installed_ollama_models
]

if missing_ollama_models:
    raise ValueError(f"Missing Ollama models: {missing_ollama_models}")

if any(model_config["provider"] == "openai" for model_config in MODEL_CONFIGS):
    if not os.environ.get("OPENAI_API_KEY"):
        raise ValueError("Missing OPENAI_API_KEY in .env")

print("Installed Ollama models:", installed_ollama_models)
print("OpenAI key loaded:", bool(os.environ.get("OPENAI_API_KEY")))

Installed Ollama models: ['gemma2:9b', 'llama3.1:latest']
OpenAI key loaded: True


## Load Validation Examples

In [26]:
def load_xsum_examples(limit):
    xsum = load_dataset("EdinburghNLP/xsum", split=f"validation[:{limit}]")

    examples = []
    for row in xsum:
        examples.append(
            {
                "id": row["id"],
                "dataset": "xsum",
                "article": row["document"],
                "reference_summary": row["summary"],
            }
        )

    return examples


def load_cnn_dailymail_examples(limit):
    if not CNN_DM_VALIDATION_FILE.exists():
        raise FileNotFoundError(
            f"Missing CNN/DailyMail validation file: {CNN_DM_VALIDATION_FILE}\n"
            "Download the Kaggle dataset before running this cell."
        )

    df = pd.read_csv(CNN_DM_VALIDATION_FILE, nrows=limit)

    examples = []
    for row in df.to_dict(orient="records"):
        examples.append(
            {
                "id": row["id"],
                "dataset": "cnn_dailymail",
                "article": row["article"],
                "reference_summary": row["highlights"],
            }
        )

    return examples

In [27]:
xsum_examples = load_xsum_examples(NUM_EXAMPLES_PER_DATASET)
cnn_dailymail_examples = load_cnn_dailymail_examples(NUM_EXAMPLES_PER_DATASET)
examples = xsum_examples + cnn_dailymail_examples

df_examples = pd.DataFrame(examples)
df_examples[["dataset", "id", "article", "reference_summary"]]

,dataset,id,article,reference_summary
0,xsum,38295789,The ex-Reading defender denied fraudulent trad...,Former Premier League footballer Sam Sodje has...
1,xsum,40202028,Voges was forced to retire hurt on 86 after su...,Middlesex batsman Adam Voges will be out until...
2,xsum,36177725,Seven photographs taken in the Norfolk country...,The Duchess of Cambridge will feature on the c...
3,xsum,35751255,"Chris Poole - known as ""moot"" online - created...",Google has hired the creator of one of the web...
4,xsum,35275743,Four police officers were injured in the incid...,Two teenagers have been charged in connection ...
...,...,...,...,...
195,cnn_dailymail,2cf347b845fee20568d174c69c4917a5e04e9481,This is the moment a diner tried to get out of...,"Christopher Baker, 28, ordered meal from Borne..."
196,cnn_dailymail,dbf81e7c9e4213a6db4e6be933fc892747084ffa,This winter has been the season that literally...,Weather map produced using data from NASA's Te...
197,cnn_dailymail,35081248964ca5fcc3ceefcdb3113c7cc6784d28,We reveal how to get the enviable physiques of...,New mother Scarlett was showing off perfect pi...
198,cnn_dailymail,17e14ffdecaa9bf007bf7cb130b503c36a49347b,Nigel Pearson has been caught on camera appear...,Leicester are bottom of the Premier League wit...


## Generate Summaries

In [28]:
def build_prompt(dataset_name, article):
    if dataset_name == "xsum":
        return f"Article:\n{article}\n\nSummarize the article in one sentence.\n\nSummary:"

    if dataset_name == "cnn_dailymail":
        return f"Article:\n{article}\n\nSummarize the article in three sentences.\n\nSummary:"

    raise ValueError(f"Unknown dataset: {dataset_name}")


def generate_ollama_summary(model_name, dataset_name, article):
    payload = {
        "model": model_name,
        "prompt": build_prompt(dataset_name, article),
        "stream": False,
        "options": {
            "temperature": TEMPERATURE,
        },
    }

    response = requests.post(OLLAMA_URL, json=payload, timeout=180)

    if response.status_code != 200:
        print(response.text)
        response.raise_for_status()

    return response.json()["response"].strip()


def extract_openai_text(response_json):
    if response_json.get("output_text"):
        return response_json["output_text"]

    text_parts = []
    for output_item in response_json.get("output", []):
        for content_item in output_item.get("content", []):
            if content_item.get("type") == "output_text":
                text_parts.append(content_item.get("text", ""))

    if text_parts:
        return "\n".join(text_parts).strip()

    raise KeyError(f"Could not find generated text in response: {response_json}")


def generate_openai_summary(model_name, dataset_name, article):
    payload = {
        "model": model_name,
        "input": build_prompt(dataset_name, article),
        "temperature": TEMPERATURE,
    }
    headers = {
        "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
        "Content-Type": "application/json",
    }

    response = requests.post(OPENAI_URL, headers=headers, json=payload, timeout=180)

    if response.status_code != 200:
        print(response.text)
        response.raise_for_status()

    return extract_openai_text(response.json())


def generate_summary(model_config, dataset_name, article):
    provider = model_config["provider"]
    model_name = model_config["model"]

    if provider == "ollama":
        return generate_ollama_summary(model_name, dataset_name, article)

    if provider == "openai":
        return generate_openai_summary(model_name, dataset_name, article)

    raise ValueError(f"Unknown provider: {provider}")

In [29]:
records = []

for model_config in MODEL_CONFIGS:
    for example in examples:
        start_time = time.time()
        generated_summary = generate_summary(model_config, example["dataset"], example["article"])
        elapsed_seconds = round(time.time() - start_time, 2)

        records.append(
            {
                "run_id": RUN_ID,
                "id": example["id"],
                "dataset": example["dataset"],
                "provider": model_config["provider"],
                "model": model_config["model"],
                "article": example["article"],
                "reference_summary": example["reference_summary"],
                "generated_summary": generated_summary,
                "elapsed_seconds": elapsed_seconds,
            }
        )

df_outputs = pd.DataFrame(records)
df_outputs[["run_id", "dataset", "provider", "model", "id", "reference_summary", "generated_summary", "elapsed_seconds"]]

,run_id,dataset,provider,model,id,reference_summary,generated_summary,elapsed_seconds
0,20260519_213553,xsum,ollama,llama3.1:latest,38295789,Former Premier League footballer Sam Sodje has...,"Four brothers, including former Reading defend...",5.21
1,20260519_213553,xsum,ollama,llama3.1:latest,40202028,Middlesex batsman Adam Voges will be out until...,Middlesex cricket team is hoping that Adam Vog...,3.49
2,20260519_213553,xsum,ollama,llama3.1:latest,36177725,The Duchess of Cambridge will feature on the c...,The Duchess of Cambridge will be featured in t...,3.50
3,20260519_213553,xsum,ollama,llama3.1:latest,35751255,Google has hired the creator of one of the web...,"Chris Poole, also known as ""moot,"" who previou...",3.82
4,20260519_213553,xsum,ollama,llama3.1:latest,35275743,Two teenagers have been charged in connection ...,"Two teenagers, aged 16 and 19, have been charg...",3.24
...,...,...,...,...,...,...,...,...
595,20260519_213553,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,2cf347b845fee20568d174c69c4917a5e04e9481,"Christopher Baker, 28, ordered meal from Borne...","Christopher Baker, 28, attempted to avoid payi...",3.16
596,20260519_213553,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,dbf81e7c9e4213a6db4e6be933fc892747084ffa,Weather map produced using data from NASA's Te...,This winter showcased a stark temperature divi...,2.50
597,20260519_213553,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,35081248964ca5fcc3ceefcdb3113c7cc6784d28,New mother Scarlett was showing off perfect pi...,The article discusses Scarlett Johansson's imp...,3.36
598,20260519_213553,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,17e14ffdecaa9bf007bf7cb130b503c36a49347b,Leicester are bottom of the Premier League wit...,Leicester City manager Nigel Pearson was caugh...,3.13


## Save Generated Summaries

In [30]:
with OUTPUT_FILE.open("w", encoding="utf-8") as f:
    for record in records:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")

OUTPUT_FILE

WindowsPath('C:/Users/fangw/162 proj/outputs/20260519_213553_initial_summaries.jsonl')

## Compute Automatic Metrics

The paper reports ROUGE-L, METEOR, BERTScore, BLEURT, and BARTScore. This notebook computes the lightweight metrics first: ROUGE, METEOR, and BERTScore. BLEURT and BARTScore are heavier optional follow-ups because they require extra model/checkpoint setup.

In [31]:
class NoWordNet:
    def synsets(self, word):
        return []


scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
no_wordnet = NoWordNet()

score_rows = []
for record in records:
    scores = scorer.score(record["reference_summary"], record["generated_summary"])
    meteor = single_meteor_score(
        record["reference_summary"].split(),
        record["generated_summary"].split(),
        wordnet=no_wordnet,
    )

    score_rows.append(
        {
            "run_id": record["run_id"],
            "id": record["id"],
            "dataset": record["dataset"],
            "provider": record["provider"],
            "model": record["model"],
            "rouge1_fmeasure": scores["rouge1"].fmeasure,
            "rouge2_fmeasure": scores["rouge2"].fmeasure,
            "rougeL_fmeasure": scores["rougeL"].fmeasure,
            "meteor": meteor,
        }
    )

df_scores = pd.DataFrame(score_rows)
df_scores

,run_id,id,dataset,provider,model,rouge1_fmeasure,rouge2_fmeasure,rougeL_fmeasure,meteor
0,20260519_213553,38295789,xsum,ollama,llama3.1:latest,0.297872,0.044444,0.212766,0.136612
1,20260519_213553,40202028,xsum,ollama,llama3.1:latest,0.285714,0.074074,0.250000,0.303486
2,20260519_213553,36177725,xsum,ollama,llama3.1:latest,0.500000,0.280000,0.500000,0.456031
3,20260519_213553,35751255,xsum,ollama,llama3.1:latest,0.285714,0.042553,0.122449,0.119760
4,20260519_213553,35275743,xsum,ollama,llama3.1:latest,0.408163,0.170213,0.367347,0.312153
...,...,...,...,...,...,...,...,...,...
595,20260519_213553,2cf347b845fee20568d174c69c4917a5e04e9481,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,0.431138,0.157576,0.275449,0.294196
596,20260519_213553,dbf81e7c9e4213a6db4e6be933fc892747084ffa,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,0.250000,0.031746,0.125000,0.128440
597,20260519_213553,35081248964ca5fcc3ceefcdb3113c7cc6784d28,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,0.160000,0.000000,0.120000,0.107239
598,20260519_213553,17e14ffdecaa9bf007bf7cb130b503c36a49347b,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,0.337500,0.088608,0.187500,0.178656


## Compute BERTScore

BERTScore downloads a model the first time it runs. For this small test set, CPU execution is fine but may take a little while.

In [32]:
generated_summaries = [record["generated_summary"] for record in records]
reference_summaries = [record["reference_summary"] for record in records]

bert_precision, bert_recall, bert_f1 = bert_score(
    generated_summaries,
    reference_summaries,
    lang="en",
    verbose=True,
)

df_scores["bertscore_precision"] = bert_precision.tolist()
df_scores["bertscore_recall"] = bert_recall.tolist()
df_scores["bertscore_f1"] = bert_f1.tolist()

df_scores

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/13 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/10 [00:00<?, ?it/s]

done in 73.40 seconds, 8.17 sentences/sec


,run_id,id,dataset,provider,model,rouge1_fmeasure,rouge2_fmeasure,rougeL_fmeasure,meteor,bertscore_precision,bertscore_recall,bertscore_f1
0,20260519_213553,38295789,xsum,ollama,llama3.1:latest,0.297872,0.044444,0.212766,0.136612,0.897731,0.921923,0.909666
1,20260519_213553,40202028,xsum,ollama,llama3.1:latest,0.285714,0.074074,0.250000,0.303486,0.870993,0.903162,0.886786
2,20260519_213553,36177725,xsum,ollama,llama3.1:latest,0.500000,0.280000,0.500000,0.456031,0.886862,0.939822,0.912574
3,20260519_213553,35751255,xsum,ollama,llama3.1:latest,0.285714,0.042553,0.122449,0.119760,0.858459,0.906577,0.881862
4,20260519_213553,35275743,xsum,ollama,llama3.1:latest,0.408163,0.170213,0.367347,0.312153,0.903280,0.922456,0.912767
...,...,...,...,...,...,...,...,...,...,...,...,...
595,20260519_213553,2cf347b845fee20568d174c69c4917a5e04e9481,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,0.431138,0.157576,0.275449,0.294196,0.882931,0.877128,0.880020
596,20260519_213553,dbf81e7c9e4213a6db4e6be933fc892747084ffa,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,0.250000,0.031746,0.125000,0.128440,0.848531,0.840013,0.844250
597,20260519_213553,35081248964ca5fcc3ceefcdb3113c7cc6784d28,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,0.160000,0.000000,0.120000,0.107239,0.834605,0.849154,0.841817
598,20260519_213553,17e14ffdecaa9bf007bf7cb130b503c36a49347b,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,0.337500,0.088608,0.187500,0.178656,0.875312,0.858370,0.866759


In [33]:
df_scores.to_csv(RESULTS_FILE, index=False)

df_combined = df_outputs.merge(
    df_scores,
    on=["run_id", "id", "dataset", "provider", "model"],
    how="left",
)
df_combined.to_csv(COMBINED_RESULTS_FILE, index=False)

df_scores.groupby(["dataset", "provider", "model"])[
    [
        "rouge1_fmeasure",
        "rouge2_fmeasure",
        "rougeL_fmeasure",
        "meteor",
        "bertscore_f1",
    ]
].mean().reset_index()

,dataset,provider,model,rouge1_fmeasure,rouge2_fmeasure,rougeL_fmeasure,meteor,bertscore_f1
0,cnn_dailymail,ollama,gemma2:9b,0.379870,0.120905,0.232404,0.244368,0.872632
1,cnn_dailymail,ollama,llama3.1:latest,0.397178,0.139147,0.233935,0.294945,0.874908
2,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,0.375277,0.122804,0.226897,0.249384,0.873190
3,xsum,ollama,gemma2:9b,0.288234,0.092956,0.222352,0.196276,0.891055
4,xsum,ollama,llama3.1:latest,0.283640,0.079827,0.206246,0.216452,0.887343
5,xsum,openai,gpt-4o-mini-2024-07-18,0.271394,0.071133,0.192551,0.192729,0.884453


## Exported Files

In [34]:
print(f"Generated summaries JSONL: {OUTPUT_FILE}")
print(f"Metric scores CSV: {RESULTS_FILE}")
print(f"Combined summaries + metrics CSV: {COMBINED_RESULTS_FILE}")

df_combined[[
    "run_id",
    "dataset",
    "provider",
    "model",
    "id",
    "reference_summary",
    "generated_summary",
    "rougeL_fmeasure",
    "meteor",
    "bertscore_f1",
]]

Generated summaries JSONL: C:\Users\fangw\162 proj\outputs\20260519_213553_initial_summaries.jsonl
Metric scores CSV: C:\Users\fangw\162 proj\results\20260519_213553_initial_metric_scores.csv
Combined summaries + metrics CSV: C:\Users\fangw\162 proj\results\20260519_213553_initial_summaries_with_metrics.csv


,run_id,dataset,provider,model,id,reference_summary,generated_summary,rougeL_fmeasure,meteor,bertscore_f1
0,20260519_213553,xsum,ollama,llama3.1:latest,38295789,Former Premier League footballer Sam Sodje has...,"Four brothers, including former Reading defend...",0.212766,0.136612,0.909666
1,20260519_213553,xsum,ollama,llama3.1:latest,40202028,Middlesex batsman Adam Voges will be out until...,Middlesex cricket team is hoping that Adam Vog...,0.250000,0.303486,0.886786
2,20260519_213553,xsum,ollama,llama3.1:latest,36177725,The Duchess of Cambridge will feature on the c...,The Duchess of Cambridge will be featured in t...,0.500000,0.456031,0.912574
3,20260519_213553,xsum,ollama,llama3.1:latest,35751255,Google has hired the creator of one of the web...,"Chris Poole, also known as ""moot,"" who previou...",0.122449,0.119760,0.881862
4,20260519_213553,xsum,ollama,llama3.1:latest,35275743,Two teenagers have been charged in connection ...,"Two teenagers, aged 16 and 19, have been charg...",0.367347,0.312153,0.912767
...,...,...,...,...,...,...,...,...,...,...
595,20260519_213553,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,2cf347b845fee20568d174c69c4917a5e04e9481,"Christopher Baker, 28, ordered meal from Borne...","Christopher Baker, 28, attempted to avoid payi...",0.275449,0.294196,0.880020
596,20260519_213553,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,dbf81e7c9e4213a6db4e6be933fc892747084ffa,Weather map produced using data from NASA's Te...,This winter showcased a stark temperature divi...,0.125000,0.128440,0.844250
597,20260519_213553,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,35081248964ca5fcc3ceefcdb3113c7cc6784d28,New mother Scarlett was showing off perfect pi...,The article discusses Scarlett Johansson's imp...,0.120000,0.107239,0.841817
598,20260519_213553,cnn_dailymail,openai,gpt-4o-mini-2024-07-18,17e14ffdecaa9bf007bf7cb130b503c36a49347b,Leicester are bottom of the Premier League wit...,Leicester City manager Nigel Pearson was caugh...,0.187500,0.178656,0.866759
